# Machine Learning for Signal Processing — Homework 2
### Representation and Structure | Weeks 4–5
**Amirkabir University of Technology · Department of Electrical Engineering**

---

Module II assignment.  

Notebook Author: Parisa Absavaran

Teaching Assistant — Machine Learning for Signal Processing
Amirkabir University of Technology
___

*If you have any questions, please contact the TAs.* 
___

This assignment comprises four connected signal-processing problems. In each problem, begin with the observation model and its structural assumptions, implement the specified numerical methods, and evaluate the results under controlled changes in experimental conditions. All required derivations, code, tables, plots and interpretations belong in this notebook. No separate report is needed.

**Completion convention.** Implement `TODO-FUNCTION` blocks and complete `TODO-EXPERIMENT` cells. Retain the given input/output conventions, random seeds and test cells. The TA reference notebook contains completed solutions and must not be distributed to students.

**Permitted tools.** NumPy, SciPy, Matplotlib, pandas, dataset/splitting/metric utilities, and `LogisticRegression` for the downstream task are permitted. Do not replace the required EVD/SVD-based PCA, OMP or ISTA implementations with ready-made estimators. The supplied `soft_threshold` function is a shared helper; implement the ISTA update yourself. All fitting and hyperparameter selection must occur without access to held-out test outcomes.

Before submission, run **Kernel → Restart & Run All** and check that all required tests pass. Optional extensions are not graded.

## 0 — Setup, conventions and reusable diagnostics
We store samples as rows: `X.shape == (N,d)`; this differs from the lecturer's column-sample convention but is algebraically equivalent when used consistently. Covariance uses the **1/N** convention for compatibility with the course derivation. `NMSE = ||x̂−x||²/||x||²` is computed against nonzero ground truth. A *projection matrix* is invariant to sign flips and rotations within a principal subspace.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import subspace_angles
from scipy.fft import dct, idct
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from skimage.data import lfw_subset
from time import perf_counter
SEED = 2026
np.set_printoptions(precision=4,suppress=True)
plt.rcParams.update({'figure.figsize':(7,4),'axes.grid':True,'grid.alpha':.2})
def normalize_columns(A):
    norms = np.linalg.norm(A,axis=0)
    if np.any(norms < 1e-12): raise ValueError('zero-norm column')
    return A / norms

def nmse(x, xhat):
    denom=np.sum(np.asarray(x)**2)
    if denom<=1e-15: raise ValueError('NMSE requires nonzero reference')
    return np.sum((np.asarray(x)-np.asarray(xhat))**2)/denom

def support_metrics(true,estimated):
    t,e=set(true),set(estimated)
    tp=len(t&e)
    return tp/max(len(e),1),tp/max(len(t),1),t==e

def fmt(mean_vals): return np.mean(mean_vals),np.std(mean_vals,ddof=1) if len(mean_vals)>1 else 0.0
print('Environment ready; seed:',SEED)

# Provided scalar/vector proximal operator (not graded)
def soft_threshold(v,tau):
    v=np.asarray(v,dtype=float)
    return np.sign(v)*np.maximum(np.abs(v)-tau,0.0)


# Question 1 — Recovering structure from noisy multi-sensor measurements (25 points)

A measurement system uses eight sensors to observe a process driven by two latent sources. Although the observed vector belongs to an eight-dimensional space, the clean signal lies in a two-dimensional linear subspace. Each observation is corrupted by independent Gaussian sensor noise:

$$Y=ZB^T+\sigma E,\qquad B\in\mathbb R^{8\times2},\quad B^TB=I_2.$$

Here rows correspond to observations, $Z$ contains the two latent source coefficients, and the entries of $E$ are independent standard normal variables. The measurement matrix and the random seed used to generate the initial data are supplied below. The aim is to estimate the signal subspace, reconstruct the observations with a reduced rank, and determine when the estimated directions can be trusted.

### 1.1 — PCA through eigendecomposition and SVD (6 points)
Complete `pca_evd` and `pca_svd` using the sample covariance convention $C=X_c^TX_c/N$. Return the fitted mean, principal basis and descending eigenvalue spectrum. Verify the relation between the two decompositions and compare subspaces through $UU^T$ rather than signs of individual eigenvectors. Derive the maximum-variance formulation of PCA with a Lagrange multiplier.

> **Useful references (optional):**
> - [Applied Linear Algebra (Boyd & Vandenberghe)](https://www.web.stanford.edu/~boyd/vmls/)
> - [SciPy Principal Angles Documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.subspace_angles.html)

In [ ]:
rng=np.random.default_rng(SEED+1)
N,d,r=450,8,2
B,_=np.linalg.qr(rng.normal(size=(d,r)))
latent=rng.normal(size=(N,r))*np.array([3.0,0.9])
clean=latent @ B.T
noise_sigma=.55
observed=clean+noise_sigma*rng.normal(size=(N,d))
print('Observed:',observed.shape,'True subspace:',B.shape)

### 1.1 — Regularized PCA (6 points)

Use the centered covariance `C = Xc.T @ Xc / N` and diagonal-target shrinkage `C_gamma = (1-gamma)*C + gamma*diag(diag(C))`. Implement `regularized_pca(X,k,gamma)`, retaining the original EVD and SVD functions as reusable numerical tools for the subsequent sections. Compare `gamma ∈ {0,0.1,0.3}` using subspace angles and reconstruction on a held-out simulation. **(a) Implementation: 3 pts; (b) comparison: 2 pts; (c) interpretation: 1 pt.**

In [ ]:
def pca_evd(X,k):
    # TODO-FUNCTION: mean, descending covariance eigenpairs
    raise NotImplementedError

def pca_svd(X,k):
    # TODO-FUNCTION: mean, descending right singular vectors and variance
    raise NotImplementedError

def regularized_pca(X,k,gamma):
    # TODO-FUNCTION: centered diagonal-target covariance shrinkage
    raise NotImplementedError



In [ ]:
me,Ue,ve=pca_evd(observed,2);ms,Us,vs=pca_svd(observed,2)
assert Ue.shape==Us.shape==(d,2)
assert np.allclose(ve,vs,rtol=1e-8,atol=1e-9)
assert np.linalg.norm(Ue@Ue.T-Us@Us.T)<1e-8
print('Subspace projector difference:',np.linalg.norm(Ue@Ue.T-Us@Us.T))
# Regularized PCA diagnostic (not a substitute for the required analysis)
mu_reg,U_reg,eig_reg=regularized_pca(observed,2,0.0)
assert U_reg.shape==(d,2) and np.all(np.diff(eig_reg)<=1e-10)


**Analysis 1.1:** Discuss the behavior of diagonal-target covariance shrinkage, and why a held-out evaluation is needed. No single gamma is guaranteed to improve the subspace estimate.

**Your answer 1.1:**  
TODO-ANALYSIS: record the numerical comparisons and interpretation.

**1.2 (6 points)** Reconstruct the observations with ranks `k=1,…,8` and compare (i) reconstruction error against noisy data, (ii) denoising error against the **clean** signal, and (iii) residual singular energy from Eckart–Young. Discuss why the best rank for denoising differs from the best rank for fitting observed data.

### 1.2 — Projection and reconstruction

Fit each rank on `observed`. Calculate all three curves and draw a single readable figure.

In [ ]:
def pca_reconstruct(X,mu,U):
    # TODO-FUNCTION
    raise NotImplementedError
# TODO-EXPERIMENT: implement the three curves and plot them.

**1.3 (8 points)** Scientific stability study. Keep the true latent subspace fixed. Vary sample size `N ∈ {30,80,250}` and noise standard deviation `σ ∈ {0.2,0.8,1.5}`. For each configuration, generate **25 independent trials**, fit a two-dimensional PCA subspace and report mean ± standard deviation of the **largest principal angle** to the known true subspace. Show a heatmap and explain the role of the eigengap, noise and sample size. Use the same latent strengths across configurations; do not change the target while comparing methods.

### 1.3 — Monte Carlo subspace-angle experiment

Implement repeated trials with independent random streams. `subspace_angles` returns angles in radians.

In [ ]:
# TODO-EXPERIMENT: 3 × 3 configurations × 25 trials.
# Report mean and SD; visualize mean maximum principal angle.

**1.4 (5 points)** Derive the identity `||Xc−Xc,k||²_F = ∑_{i>k} sᵢ²`. Repeat the rank-2 fit after (a) omitting centering and (b) multiplying one feature by 10. Explain the differences as **modeling choices**, not implementation bugs. State when standardization is appropriate for physical sensors with different units.

In [ ]:
# TODO-EXPERIMENT (TA reference): centering and scaling effects
raw_U=np.linalg.svd(observed,full_matrices=False)[2][:2].T
_,cen_U,_=pca_svd(observed,2)
scaled=observed.copy();scaled[:,0]*=10
_,sc_U,_=pca_svd(scaled,2)
print('Uncentered vs centered projector gap:',np.linalg.norm(raw_U@raw_U.T-cen_U@cen_U.T))
print('Scaled first-feature loading magnitude:',np.abs(sc_U[0]))

# Question 2 — Does a compact representation preserve useful information? (25 points)

A reduced representation may reconstruct an observed signal accurately without preserving the features required for a downstream decision. Investigate this distinction in two settings. First, use a collection of small *unlabeled* face images to study reconstruction through Eigenfaces. Second, use labeled images of handwritten digits to measure how dimensionality reduction affects classification. The face dataset has no identity labels: do not claim to perform identity recognition on it.

### 2.1 — Eigenfaces and held-out reconstruction (6 points)
Split face images into training/test subsets (75/25). Learn the mean face and basis **only from training images**, and compute the first five Eigenfaces. Reconstruct test images for $k\in\{1,3,5,10,20,40\}$. Report the test pixel MSE and the cumulative training explained variance. Explain why reconstruction quality on held-out images is a different quantity from training reconstruction error.

> **Useful references (optional):**
> - [scikit-image Face Image Data](https://scikit-image.org/docs/stable/api/skimage.data)
> - [scikit-learn Handwritten Digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)
> - [Train/Test Split Documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
> Note: `skimage.data.lfw_subset` includes both face and non-face images and does not provide identity labels for person recognition.

In [ ]:
faces=lfw_subset().astype(float);h,w=faces.shape[1:];face_X=faces.reshape(len(faces),-1)
Ftr,Fte=train_test_split(face_X,test_size=.25,random_state=SEED,shuffle=True)
print('Train/test faces',Ftr.shape,Fte.shape)

### 2.1 — Eigenfaces

Function returns training mean, basis and full singular spectrum. Plot mean, eigenfaces and reconstructions; do not recompute mean on test.

In [ ]:
def fit_eigenfaces(Xtrain,k):
    # TODO-FUNCTION: fit SVD on training only; return mean,basis,all singular values
    raise NotImplementedError
# TODO-EXPERIMENT: mean image, 5 eigenfaces, 6 rank reconstructions,
# held-out pixel MSE and explained-variance curve.

**2.2 (10 points)** Load the labeled **digits** dataset (8×8 grayscale). Make one stratified 60/20/20 train/validation/test split. Fit PCA **only on train** for `k ∈ {2,5,10,20,40}`. Train multinomial-capable logistic regression (fixed `C=1.0`, `max_iter=2000`) on each representation; choose `k` using validation accuracy, with smallest `k` breaking ties. Compare against a classifier on raw pixels. Only after selection, report held-out test accuracy and confusion matrix. Include validation explained variance and analyze whether the representation minimizing reconstruction error also optimizes classification.

### 2.2 — Task-aware evaluation

Training-only PCA; validation-only choice; test once; add a comparison table and one confusion matrix.

In [ ]:
# TODO-EXPERIMENT: stratified train/validation/test, PCA fit on train only,
# choose k using validation, report final test once and confusion matrix.

**2.3 (5 points)** Create two synthetic features with nuisance variance much larger than class-related variance: `x₁~N(0,10²)`, `x₂|class=±1~N(±1,0.35²)`. Use independent train and test realizations. Demonstrate how PC1 can capture nearly all variance but lose almost all classification signal. Explain which objective PCA optimizes and which objective classification requires.

### 2.3 — Controlled counterexample

Evaluate a classifier based on PC1 against a classifier using the low-variance informative feature.

In [ ]:
# TODO-EXPERIMENT: generate independent split, compare PC1 vs informative feature;
# report explained variance and held-out classification accuracy.

**2.4 (4 points)** Derive PCA whitening from `C=UΛUᵀ`, implement regularized whitening `(Λ+εI)^(-1/2)Uᵀ(x−μ)` and compare covariance diagonals for two choices of `ε`. Why does whitening small-variance noise directions cause numerical and statistical problems? Distinguish uncorrelatedness from independence.

In [ ]:
# TODO-EXPERIMENT (TA reference): whitening
mu_w,U_w,eigs_w=pca_evd(Xtr,20)
for eps in [1e-8,.05]:
    whitened=(Xtr-mu_w)@U_w/np.sqrt(eigs_w[:20]+eps)
    diag=np.diag(np.cov(whitened,rowvar=False,bias=True))
    print('epsilon',eps,'mean variance',diag.mean(),'minimum variance',diag.min())

# Question 3 — Sparse recovery, OMP, ISTA and compressed sensing (35 points)

A signal of length 64 is modeled as a linear combination of a small number of atoms from a dictionary with 128 columns. The observation model is

$$y=D\alpha+n,\qquad D\in\mathbb R^{64\times128},\quad \|\alpha\|_0=4,$$

where each dictionary column has unit Euclidean norm and $n$ denotes additive noise. Recovering the coefficients involves identifying the active atoms as well as estimating their amplitudes. These are related but distinct evaluation goals.


> **Useful references (optional):**
> - [Applied Linear Algebra and Least Squares](https://www.web.stanford.edu/~boyd/vmls/)
> - [SciPy Linear Algebra Reference](https://docs.scipy.org/doc/scipy/reference/linalg.html)
> Note: scikit-learn Lasso scales the squared-error term by `1/n_samples`; check the penalty convention before comparing methods.

**3.1 (9 points)** Implement **OMP from scratch** for normalized dictionaries. At each iteration choose the *unused* column with the largest absolute residual correlation; refit **all** selected coefficients using `np.linalg.lstsq`; update residual. Stop at a target cardinality, low residual, or no informative unused column. Return coefficient vector, selected indices and the complete residual-norm history. Do not use `sklearn.linear_model.OrthogonalMatchingPursuit`. For `D∈R^(64×128)` with 4 nonzero coefficients, plot selection order and residual decay.

### 3.1 — OMP implementation

Return `(alpha, support, history)`, with `history[0]=||y||₂`. Be explicit about the stopping tolerance.

In [ ]:
def omp(D,y,k,tol=1e-8):
    # TODO-FUNCTION: normalized columns, never select an atom twice,
    # least-squares refit over entire selected support, residual-norm history.
    raise NotImplementedError
# TODO-EXPERIMENT: normalized 64 × 128 dictionary, 4-sparse truth;
# compare support and plot OMP residual decay.

In [ ]:
Dcheck=np.eye(5);a,sel,h=omp(Dcheck,np.array([0.,3.,0.,-2.,0.]),2)
assert set(sel)=={1,3} and np.allclose(a,[0,3,0,-2,0])
assert len(h)==len(sel)+1 and np.all(np.diff(h)<=1e-7)
print('OMP orthogonal-dictionary test passed.')

**3.2 (7 points)** Implement ISTA for `F(alpha)=0.5||A@alpha-y||² + lambda||alpha||_1` with a stable step `1/L`, where `L >= ||A||_2²`. Return both the recovered coefficients and the objective history; plot objective versus iteration. For the final OMP/ISTA comparison, use `n=96, m=48, k=4`, `SNR={0,10,20}` dB, and at least **30 paired trials per SNR** (both methods receive each same noisy observation). Tune `lambda` on **independent validation observations**, then freeze it for all test trials. Report **mean and standard deviation** of coefficient NMSE, support precision, support recall, exact-support recovery and runtime, with an interpretation of failures. **ISTA implementation and objective plot: 3 pts; validation design and paired trials: 2 pts; statistical reporting and interpretation: 2 pts.**

### 3.2 — OMP versus ISTA (7 points)

For the comparison use independent validation data and `n=96, m=48, k=4`. Use exactly the same sensing observation for the two methods in each paired test trial. For ISTA support, declare and document a fixed numerical threshold. Report mean **and standard deviation** of all requested metrics.

In [ ]:
def ista(A,y,lam,n_iter=300,tol=1e-7):
    # TODO-FUNCTION: proximal-gradient step, objective history and stopping
    raise NotImplementedError

# TODO-EXPERIMENT (3.2): plot ISTA objective trace; choose lambda on independent validation.
# Compare OMP and ISTA for n=96, m=48, k=4, SNR={0,10,20} dB.
# At least 30 paired test trials at each SNR; report mean/std of coefficient NMSE,
# support precision/recall, exact support, and runtime for BOTH methods.


**3.3 (5 points)** Construct a *controlled high-coherence dictionary* by replacing one atom with a near-copy of another, then renormalizing. Compare its mutual coherence to an ordinary Gaussian dictionary. Over **50** independent sparse/noisy observations, record exact-support recovery and explain why similar atom pairs cause ambiguity. Keep dictionary size, target sparsity, SNR and evaluation rules fixed. Avoid claiming that one observed failure establishes a universal theorem.

### 3.3 — Coherence stress test

Use `max |DᵀD−I|` for normalized columns and choose a support that sometimes includes the ambiguous pair.

In [ ]:
def mutual_coherence(D):
    # TODO-FUNCTION
    raise NotImplementedError
# TODO-EXPERIMENT: ordinary vs near-duplicate dictionary, ≥50 trials,
# controlled support design, exact-support recovery rates.

## Question 3 — Compressed sensing experiments (continued)

> **Useful references (optional):**
> - [SciPy Orthonormal DCT](https://docs.scipy.org/doc/scipy/reference/generated/scipy.fft.dct.html)
> - [SciPy Linear Algebra Reference](https://docs.scipy.org/doc/scipy/reference/linalg.html)

### 3.4 — Controlled compressed-sensing generator

Return `(A, x, y, y_clean)`; do not silently rescale ground truth after constructing it.

In [ ]:
def make_cs_problem(n,m,k,snr_db,seed):
    # TODO-FUNCTION: normalized Gaussian sensing matrix,
    # exactly k nonzeros, signal-power SNR convention.
    raise NotImplementedError

In [ ]:
A,x,y,y_clean=make_cs_problem(96,32,4,20,SEED)
assert A.shape==(32,96) and y.shape==(32,) and np.count_nonzero(x)==4
assert np.allclose(np.linalg.norm(A,axis=0),1,atol=1e-8)
print('CS generator tests passed.')

**3.5 (9 points)** Phase diagram. Fix `n=96`, use `m∈{16,24,32,48,64}` and `k∈{2,4,6,8,12}`; for each cell run **at least 30 independent trials** with noise-free observations. OMP is given the true `k` *only in this controlled experiment* (oracle stopping budget). Define success using both `NMSE<10⁻³` and exact support; plot **two separate heatmaps**. Report a binomial uncertainty interval or standard error for success rates, and interpret the empirical transition without treating `m≈k log(n/k)` as an exact law.

### 3.5 — Monte Carlo phase transition

Implement nested loops with deterministic independent trial seeds; return two success maps and their standard errors.

In [ ]:
def recovery_grid(n=96,ms=(16,24,32,48,64),ks=(2,4,6,8,12),trials=30):
    # TODO-FUNCTION: independent Monte Carlo per (k,m),
    # BOTH NMSE and exact-support success rates.
    raise NotImplementedError
# TODO-EXPERIMENT: two labeled heatmaps and binomial uncertainty.

# Question 4 — Missing-sample recovery after Burst Loss (15 points)

Original DCT-based recovery task, with 40% random missing samples versus one continuous missing signal block.

**4.1 (15 points)** Missing-sample signal recovery. Construct a real 128-point signal with exactly **five nonzero orthonormal DCT coefficients**. Create **two masks with the same 40% missing rate**: (i) random missing samples and (ii) one contiguous missing signal block (Burst Loss); write `y=M Ψ α + n`. Solve for α with OMP using the **column-normalized effective matrix** and correctly undo normalization when reconstructing. Compare with zero-fill on **missing-sample MSE** (do not assess only observed entries). For **both masks**, report missing-sample MSE and full-signal NMSE for OMP and zero-fill. Explain the role of the effective sensing matrix and why the two missing patterns may differ. Distinguish the transform dictionary Ψ from the measurement operator M.
**Grading:** signal and both masks 3 pts; effective sensing matrix/OMP recovery 5 pts; missing-sample MSE and NMSE versus zero-fill 3 pts; interpretation 4 pts.

### 4.1 — Random Missing Samples versus Burst Loss (15 points)

Use exactly the same number of missing samples in both masks. The effective sensing matrix is formed by selecting observed rows of the DCT basis. Renormalize its nonzero columns for OMP, and undo that normalization before signal reconstruction. Compare missing-sample MSE and full-signal NMSE against zero-fill for both masks.

In [ ]:
# TODO-EXPERIMENT (4.1): build a 128-sample, 5-sparse orthonormal DCT signal.
# Create TWO observation masks with the SAME 40% missing rate:
# (a) random missing samples; (b) one contiguous missing signal block (Burst Loss).
# For EACH mask, form the effective sensing matrix from observed DCT rows,
# normalize columns for OMP and undo the scaling when reconstructing.
# Compare OMP recovery and zero-fill using missing-sample MSE AND full-signal NMSE.
# Report and interpret both missing patterns in a common table/figure.


# Optional investigations — for interested students (ungraded extensions)

These sections extend the same structural models; none is needed to complete Questions 1–4. They are intentionally left open-ended. Record the model, assumptions, numerical method and a brief interpretation for each attempted investigation.

- **O1 — Regularized CCA:** extract shared linear structure from two sensor views generated by common latent sources.
- **O2 — Low-rank and sparse contamination:** distinguish a low-rank signal matrix from sparse outliers.
- **O3 — ISTA preview:** solve an $\ell_1$-penalized inverse problem using a gradient step and soft thresholding; examine objective descent.
- **O4 — Reproducibility:** repeat a smaller sparse-recovery experiment with several independent seeds and report variation in success rate and runtime.

> **Useful references (optional):**
> - [Applied Linear Algebra and Low-Rank Methods](https://www.web.stanford.edu/~boyd/vmls/)

# Bonus A — Non-negative Matrix Factorization (NMF) (+10 points)
Consider a non-negative magnitude spectrogram or a synthetic non-negative feature matrix `V≈WH`, where `W≥0` and `H≥0`.
**(a) 3 pts:** state why non-negativity leads to a parts-based representation and why ordinary PCA components need not be non-negative. **(b) 4 pts:** implement multiplicative updates for the Frobenius-loss NMF objective with a small denominator floor; use a non-negative synthetic `V` generated from two non-negative factors. **(c) 3 pts:** plot objective versus iteration and compare reconstructed `V` at ranks 2 and 4, including one comment on scaling/permutation ambiguity. Use ≥100 iterations and at least two seeds.

**English references:** [Lee & Seung, Algorithms for NMF](https://papers.nips.cc/paper_files/paper/2000/hash/f9d1152547c0bde01830b7e8bd60024c-Abstract.html) · [scikit-learn NMF user guide](https://scikit-learn.org/stable/modules/decomposition.html#nmf) (for conceptual reference only).

In [ ]:
def nmf_mu(V,rank,n_iter=200,seed=0):
    # BONUS TODO-FUNCTION: non-negative multiplicative-update NMF, return W,H,history.
    raise NotImplementedError
# BONUS TODO-EXPERIMENT: run two seeds; compare rank 2 and rank 4.


# Bonus B — Canonical Correlation Analysis (CCA) (+5 points)
Two sensor views observe a shared low-dimensional latent process through different mixing matrices.
**(a) 2 pts:** formulate `max corr(uᵀx,vᵀy)` under the canonical variance constraints. **(b) 2 pts:** compute regularized CCA through SVD of a whitened cross-covariance matrix. **(c) 1 pt:** compare the first canonical correlation at three levels of independent sensor noise. Explain why PCA and CCA solve different representation problems.

**English references:** [scikit-learn CCA API](https://scikit-learn.org/stable/modules/generated/sklearn.cross_decomposition.CCA.html) · [Stanford VMLS](https://web.stanford.edu/~boyd/vmls/).

In [ ]:
# BONUS TODO-EXPERIMENT: generate paired views, regularized covariance blocks, whiten, SVD.


### O2 — Low-rank plus sparse corruption (optional, not graded)

In [ ]:
# OPTIONAL: synthetic low-rank matrix, sparse outliers, and an iterative
# singular-value/entrywise shrinkage decomposition.

### O3 — ISTA proximal-gradient bridge (optional, not graded)

In [ ]:
# OPTIONAL: implement 100 ISTA iterations for fixed A,y,λ;
# plot full penalized objective, checking step size 1/L.

### O4 — Cross-seed reproducibility (optional, not graded)

In [ ]:
# OPTIONAL: repeat a reduced phase-map analysis across three different global seeds;
# report variability and runtime.

## Submission checklist
- All required `TODO-FUNCTION` and `TODO-EXPERIMENT` sections completed.
- Mathematics and numerical results use consistent matrix-shape and loss-normalization conventions.
- Every plot has labeled axes and a paragraph interpreting successes **and** failures.
- Exact random seeds, trial counts, metric definitions and validation choices are reported.
- Train/validation/test boundaries are respected; test is not used to select parameters.
- `Restart & Run All` completes without errors on a CPU.
- Submit **one completed `.ipynb` file**, not the TA reference.